# iFood Case - Data Architect
## 00c - Exploracao de dados (Data Exploration)

Este notebook documenta o **processo de analise exploratoria** que antecede e justifica
as decisoes do pipeline. Enquanto o `00b_profiling` mede a **qualidade** dos dados
(o que esta errado), este notebook entende o **dominio** (como os dados se comportam
e que perguntas fazem sentido).

**Objetivo:** transformar "respondi o que o case pediu" em "explorei, entendi o dominio
e entao construi as queries com fundamento".

**Cada secao termina com a DECISAO que a evidencia sustenta.**

Pre-requisito: `00_setup` e `01_ingestion` executados (tabela Bronze disponivel).
Algumas secoes finais usam a Silver, entao idealmente rode apos `02_silver`.


## 0. Setup: imports e estilo visual

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pyspark.sql import functions as F

# Paleta iFood
IFOOD_RED = "#EA1D2C"
DARK      = "#1A1A1A"
GRAY      = "#8A8A8A"
LGRAY     = "#F4F4F4"
GREEN     = "#1B7A3E"
AMBER     = "#B45309"
BLUE      = "#1565C0"

plt.rcParams.update({
    "figure.figsize": (11, 5),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": GRAY,
    "axes.labelcolor": DARK,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.titlecolor": DARK,
    "xtick.color": GRAY,
    "ytick.color": GRAY,
    "font.size": 11,
})

BRONZE = "ifood_case.bronze.yellow_tripdata_raw"
SILVER = "ifood_case.silver.yellow_trips"
print("Setup ok.")

---
## 1. Visao geral: dimensoes do dataset

Primeira pergunta de qualquer exploracao: com quantos dados estou lidando e quais colunas existem?

In [ ]:
df = spark.table(BRONZE)

print(f"Linhas: {df.count():,}")
print(f"Colunas: {len(df.columns)}")
print()
print("Schema:")
df.printSchema()

**Leitura:** 16,2 milhoes de linhas e 19 colunas de origem (mais `_ingestao_ts` de linhagem).
O volume confirma a escolha de PySpark + Delta: nao e um dataset que cabe confortavelmente em pandas.

---
## 2. Quem sao os provedores (VendorID)?

O profiling mostrou 3 vendors distintos. Mas qual o peso de cada um? Isso importa para
saber se ha vies de algum provedor dominando o dataset.

In [ ]:
vendor_dist = (df.groupBy("VendorID")
               .count()
               .orderBy(F.desc("count"))
               .toPandas())
vendor_dist["pct"] = (vendor_dist["count"] / vendor_dist["count"].sum() * 100).round(1)
display(vendor_dist)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(vendor_dist["VendorID"].astype(str), vendor_dist["count"], color=IFOOD_RED, width=0.5)
for b, v, p in zip(bars, vendor_dist["count"], vendor_dist["pct"]):
    ax.annotate(f"{v:,.0f}\n({p}%)", (b.get_x()+b.get_width()/2, v),
                textcoords="offset points", xytext=(0,5), ha="center", fontsize=10, fontweight="bold")
ax.set_title("Distribuicao de corridas por VendorID")
ax.set_xlabel("VendorID"); ax.set_ylabel("Corridas")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f"{int(v/1e6)}M"))
plt.tight_layout(); plt.show()

**DECISAO sustentada:** os 3 vendors representam provedores de tecnologia (TPEP), nao usuarios.
Como ha concentracao em poucos provedores, classifico `vendor_id` como IDENTIFIER/INTERNAL no catalogo
de governanca: e um dado cadastral que nao deve ser exposto externamente sem necessidade,
para evitar analises de concentracao de mercado entre concorrentes.

---
## 3. Distribuicao de passenger_count

A Pergunta 2 do case e sobre media de passageiros. Antes de respondê-la, preciso entender
como essa coluna se distribui - inclusive os valores problematicos que o profiling apontou.

In [ ]:
import pandas as pd
pax_dist = (df.groupBy("passenger_count")
            .count()
            .orderBy("passenger_count")
            .toPandas())
display(pax_dist)

# Grafico com destaque para valores invalidos (nulos e zero)
fig, ax = plt.subplots(figsize=(11, 5))
cores = []
labels = []
valores = []
for _, row in pax_dist.iterrows():
    pc = row["passenger_count"]
    valores.append(row["count"])
    if pc is None or (isinstance(pc, float) and pd.isna(pc)):
        labels.append("nulo"); cores.append(AMBER)
    elif pc == 0:
        labels.append("0"); cores.append(IFOOD_RED)
    else:
        labels.append(str(int(pc))); cores.append("#F5B7B1")

bars = ax.bar(range(len(valores)), valores, color=cores, width=0.7)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels)
ax.set_title("Distribuicao de passenger_count (vermelho/ambar = invalidos)")
ax.set_xlabel("Passageiros por corrida"); ax.set_ylabel("Corridas")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f"{int(v/1e6)}M" if v>=1e6 else f"{int(v/1000)}k"))
plt.tight_layout(); plt.show()

**DECISAO sustentada:** a esmagadora maioria das corridas tem 1 passageiro - o que ja antecipa
o insight de "modal individual". Os valores nulos (428.665) e zero (273.481) sao claramente invalidos:
nao existe corrida com zero passageiros. Isso justifica os filtros `passenger_count IS NOT NULL`
e `passenger_count > 0` na Silver. A cauda de 7-9 passageiros e pequena mas plausivel (vans).

---
## 4. Distribuicao de total_amount

A Pergunta 1 e sobre valor. Preciso entender a faixa de valores, os outliers e os negativos
que o profiling detectou (min -982,95).

In [ ]:
# Estatisticas descritivas
stats = df.select(
    F.min("total_amount").alias("min"),
    F.expr("percentile_approx(total_amount, 0.25)").alias("p25"),
    F.expr("percentile_approx(total_amount, 0.50)").alias("mediana"),
    F.expr("percentile_approx(total_amount, 0.75)").alias("p75"),
    F.expr("percentile_approx(total_amount, 0.99)").alias("p99"),
    F.max("total_amount").alias("max"),
    F.avg("total_amount").alias("media"),
).toPandas()
display(stats)

# Quantos registros sao negativos vs zero vs positivos
faixas = df.select(
    F.sum(F.when(F.col("total_amount") < 0, 1).otherwise(0)).alias("negativos"),
    F.sum(F.when(F.col("total_amount") == 0, 1).otherwise(0)).alias("zero"),
    F.sum(F.when(F.col("total_amount") > 0, 1).otherwise(0)).alias("positivos"),
).toPandas()
display(faixas)

In [ ]:
# Histograma da faixa valida (0 a p99) para enxergar a forma real da distribuicao
p99 = float(stats["p99"].iloc[0])
valido = (df.filter((F.col("total_amount") >= 0) & (F.col("total_amount") <= p99))
          .select("total_amount").toPandas())

fig, ax = plt.subplots(figsize=(11, 5))
ax.hist(valido["total_amount"], bins=50, color=IFOOD_RED, alpha=0.85)
ax.axvline(float(stats["media"].iloc[0]), color=DARK, linestyle="--", linewidth=2,
           label=f"media: US$ {float(stats['media'].iloc[0]):.2f}")
ax.axvline(float(stats["mediana"].iloc[0]), color=GREEN, linestyle="--", linewidth=2,
           label=f"mediana: US$ {float(stats['mediana'].iloc[0]):.2f}")
ax.set_title("Distribuicao de total_amount (faixa 0 a p99)")
ax.set_xlabel("total_amount (USD)"); ax.set_ylabel("Frequencia")
ax.legend()
plt.tight_layout(); plt.show()

**DECISAO sustentada:** a distribuicao e assimetrica a direita (cauda longa de corridas caras),
com media (~27,84) acima da mediana - tipico de dados de valor. Os 141.407 valores negativos
sao estornos/ajustes que puxariam a media para baixo de forma artificial, justificando o filtro
`total_amount >= 0`. Como a media e o que o case pede e ela e sensivel a outliers, reporto as duas
interpretacoes da Pergunta 1 com transparencia. O p99 separa a operacao normal dos outliers extremos.

---
## 5. Padrao temporal: distribuicao por hora do dia

A Pergunta 2 pede media por hora. Antes, exploro como o **volume** se distribui nas horas -
isso revela a dinamica de demanda e ajuda a interpretar a media de passageiros.

In [ ]:
# Volume por hora de embarque (no periodo valido)
por_hora = (df.filter((F.col("tpep_pickup_datetime") >= "2023-01-01") &
                      (F.col("tpep_pickup_datetime") < "2023-06-01"))
            .withColumn("hora", F.hour("tpep_pickup_datetime"))
            .groupBy("hora").count().orderBy("hora").toPandas())

fig, ax = plt.subplots(figsize=(11, 5))
cores = [IFOOD_RED if h in (17,18,19) else "#F5B7B1" for h in por_hora["hora"]]
ax.bar(por_hora["hora"], por_hora["count"], color=cores, width=0.8)
ax.set_title("Volume de corridas por hora do dia (todos os meses)")
ax.set_xlabel("Hora do embarque"); ax.set_ylabel("Corridas")
ax.set_xticks(range(0,24,2))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f"{int(v/1e6)}M" if v>=1e6 else f"{int(v/1000)}k"))
plt.tight_layout(); plt.show()

**DECISAO sustentada:** o volume tem dois vales (madrugada) e picos no fim da tarde (17-19h).
Esse padrao confirma que faz sentido analisar a Pergunta 2 **por hora do embarque** (pickup),
nao do desembarque. Tambem antecipa o insight central: o pico de volume (18h) precisa ser cruzado
com a ocupacao para revelar que demanda alta nao significa ocupacao alta.

---
## 6. Relacao entre distancia e valor (sanidade do dado)

Uma checagem de coerencia: o valor deveria crescer com a distancia. Se nao crescer,
ha problema estrutural nos dados.

In [ ]:
# Amostra para o scatter (16 mi e muito para plotar tudo)
amostra = (df.filter((F.col("total_amount").between(0, 200)) &
                     (F.col("trip_distance").between(0, 50)))
           .select("trip_distance", "total_amount")
           .sample(fraction=0.001, seed=42)
           .toPandas())

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(amostra["trip_distance"], amostra["total_amount"],
           s=6, alpha=0.2, color=IFOOD_RED)
ax.set_title(f"Distancia x Valor (amostra de {len(amostra):,} corridas)")
ax.set_xlabel("trip_distance (milhas)"); ax.set_ylabel("total_amount (USD)")
plt.tight_layout(); plt.show()

# Correlacao
corr = amostra["trip_distance"].corr(amostra["total_amount"])
print(f"Correlacao distancia x valor: {corr:.3f}")

**DECISAO sustentada:** existe correlacao positiva clara entre distancia e valor, como esperado.
Isso da confianca de que `total_amount` e um campo confiavel para a analise de faturamento.
A faixa horizontal proxima de valores fixos (em torno de US$ 70) revela as corridas de tarifa fixa
de/para aeroportos (JFK), um padrao real do dataset que nao e erro.

---
## 7. Comparacao antes/depois da limpeza (Bronze vs Silver)

Fecho a exploracao validando que a limpeza preservou a forma dos dados - ou seja,
que removi ruido sem distorcer a distribuicao real.

In [ ]:
# Comparar media de total_amount por mes: Bronze (suja) vs Silver (limpa)
bronze_mes = (df.filter((F.col("tpep_pickup_datetime") >= "2023-01-01") &
                        (F.col("tpep_pickup_datetime") < "2023-06-01"))
              .withColumn("ano_mes", F.date_format("tpep_pickup_datetime", "yyyy-MM"))
              .groupBy("ano_mes").agg(F.avg("total_amount").alias("media_bronze"))
              .orderBy("ano_mes").toPandas())

try:
    silver_mes = (spark.table(SILVER)
                  .groupBy("ano_mes").agg(F.avg("total_amount").alias("media_silver"))
                  .orderBy("ano_mes").toPandas())
    comp = bronze_mes.merge(silver_mes, on="ano_mes")
except Exception as e:
    print("Silver ainda nao disponivel, mostrando apenas Bronze.")
    comp = bronze_mes.copy()
    comp["media_silver"] = None

display(comp)

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(comp["ano_mes"], comp["media_bronze"], marker="o", linewidth=2.5,
        color=AMBER, label="Bronze (com ruido)")
if comp["media_silver"].notna().any():
    ax.plot(comp["ano_mes"], comp["media_silver"], marker="o", linewidth=2.5,
            color=IFOOD_RED, label="Silver (limpa)")
ax.set_title("Media de total_amount por mes: impacto da limpeza")
ax.set_ylabel("Media (USD)"); ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.3)
plt.tight_layout(); plt.show()

**DECISAO sustentada:** a limpeza ajusta o nivel da media (removendo negativos e invalidos)
mas **preserva a tendencia** de alta ao longo dos meses. Isso confirma que os filtros corrigiram
ruido sem inventar ou destruir o padrao real - a sazonalidade de +7,2% e um fenomeno genuino,
nao um artefato da limpeza.

---
## 8. Sintese: da exploracao as decisoes

| Exploracao | Evidencia | Decisao no pipeline |
|------------|-----------|---------------------|
| Volume e schema | 16,2 mi linhas, 19 colunas | PySpark + Delta (nao cabe em pandas) |
| VendorID | 3 provedores concentrados | vendor_id = IDENTIFIER/INTERNAL (governanca) |
| passenger_count | maioria = 1; nulos e zeros invalidos | filtros IS NOT NULL e > 0 na Silver |
| total_amount | assimetrica, 141k negativos | filtro >= 0; duas interpretacoes da Pergunta 1 |
| Padrao por hora | picos 17-19h | analise por hora do pickup; cruzar volume x ocupacao |
| Distancia x valor | correlacao positiva | total_amount confiavel para faturamento |
| Bronze vs Silver | tendencia preservada | limpeza removeu ruido sem distorcer o padrao |

**Conclusao:** cada query do pipeline e cada regra de limpeza tem origem em uma evidencia
exploratoria, nao apenas no enunciado do case. A exploracao sustenta as decisoes tecnicas
e da seguranca para defender cada escolha na apresentacao.
